# Dynamic topic modeling con BERTopic

Este cuaderno prepara un flujo reproducible para modelar temas dinámicos sobre los posteos de Revista Anfibia usando BERTopic, sin ejecutar celdas durante la creación.

## 1. Preparación del entorno

Instalar dependencias en el entorno del kernel si no están disponibles. `umap-learn` reciente requiere `scikit-learn>=1.6` (donde se incorporó `sklearn.utils.validation.validate_data`). Después de actualizar, reiniciar el kernel antes de continuar:

In [1]:
# %pip install --upgrade "coverage<7.4" "scikit-learn>=1.6" pandas bertopic sentence-transformers umap-learn hdbscan plotly

In [2]:
import os

In [3]:
import os
from pathlib import Path
import json
import re
import warnings

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from sklearn.decomposition import PCA
from sklearn.metrics.pairwise import cosine_similarity
from sklearn.neighbors import NearestNeighbors

warnings.filterwarnings("ignore", category=UserWarning)

PROJECT_ROOT = Path.cwd().resolve()
if PROJECT_ROOT.name == "notebooks":
    PROJECT_ROOT = PROJECT_ROOT.parent

TEXT_COLUMN = "document_text_final"
ATTRIBUTE_COLUMNS = ["tagline", "autor_names", "post_date", "category", "text_type"]
ID_COLUMNS = ["ID", "source_url", "title_text"]
USE_COLUMNS = ID_COLUMNS + ATTRIBUTE_COLUMNS + [TEXT_COLUMN]

# Rutas estándar del proyecto
DATA_PATH = PROJECT_ROOT / "data" / "processed" / "corpus_anfibia_limpio.csv"
assert DATA_PATH.exists(), f"Falta archivo de entrada: {DATA_PATH.name}"

MAX_DOCUMENTS = 4000

df = pd.read_csv(DATA_PATH, usecols=USE_COLUMNS, nrows=MAX_DOCUMENTS)

df[TEXT_COLUMN] = df[TEXT_COLUMN].fillna("").astype(str)
for col in ATTRIBUTE_COLUMNS:
    df[col] = df[col].fillna("").astype(str).str.strip()

df = df[df[TEXT_COLUMN].str.strip().ne("")].copy()
df = df.reset_index(drop=True)
df.insert(0, "doc_id", df.index)

df.shape

(3556, 10)

In [4]:
display(df[ID_COLUMNS + ATTRIBUTE_COLUMNS].head())
display(df["category"].value_counts().head(20).to_frame("documents"))

,ID,source_url,title_text,tagline,autor_names,post_date,category,text_type
0,15668,https://www.revistaanfibia.com/la-china-invisi...,La China invisible,Feria La Salada,Sebastián Hacher; Ariel Wilkis,2012-05-14 13:43:22,Política y Economía,Crónica
1,15706,https://www.revistaanfibia.com/grobocopatel-el...,El rico que se cree Steve Jobs,Gustavo Grobocopatel,Graciela Mochkofsky; Alexandre Roig,2012-05-24 13:36:12,Política y Economía,Crónica
2,15743,https://www.revistaanfibia.com/la-vaca-sagrada/,La vaca sagrada,,Josefina Licitra,2012-06-05 14:52:58,Tecnologías,
3,15695,https://www.revistaanfibia.com/los-arruinados-...,Los arruinados del Paraná,Adictos al juego,Gisele Sousa Dias,2012-06-14 10:16:11,Culturas,
4,15687,https://www.revistaanfibia.com/de-chile-tacna-...,De Chile a Tacna: una historia de carniceros y...,,Jorge Turpo Rivas,2012-06-25 14:52:05,Culturas,Crónica


,documents
category,
Política y Economía,1349
Culturas,985
Políticas de géneros,521
Otras,161
Salud,117
Ambiente,100
Tecnologías,98
Educación,75
Amor y sexo,51


In [5]:
def split_categories(value):
    parts = re.split(r"[;,]", str(value))
    return [part.strip() for part in parts if part.strip()]

category_rows = []
for doc_id, value in zip(df["doc_id"], df["category"]):
    categories = split_categories(value)
    if not categories:
        categories = ["Sin categoria"]
    for category in categories:
        category_rows.append({"doc_id": doc_id, "category_normalized": category})

categories_df = pd.DataFrame(category_rows)
category_counts = categories_df["category_normalized"].value_counts().rename_axis("category").reset_index(name="documents")

display(category_counts)

,category,documents
0,Política y Economía,1425
1,Culturas,1038
2,Políticas de géneros,539
3,Otras,165
4,Salud,126
5,Ambiente,113
6,Tecnologías,109
7,Educación,90
8,Amor y sexo,59


In [6]:
for category in category_counts["category"]:
    print(category)

Política y Economía
Culturas
Políticas de géneros
Otras
Salud
Ambiente
Tecnologías
Educación
Amor y sexo


## 2. Entrenamiento de BERTopic para modelado dinámico

Se usan los textos completos como documentos y las fechas para evaluar la evolución de los tópicos a lo largo del tiempo.

In [7]:
from pathlib import Path
import coverage.types as ct

for name, fallback in {
    'Tracer': getattr(ct, 'TTracer', object),
    'TShouldTraceFn': object,
    'TShouldStartContextFn': object,
    'TFileDisposition': object,
}.items():
    if not hasattr(ct, name):
        setattr(ct, name, fallback)

from bertopic import BERTopic
from bertopic.representation import KeyBERTInspired

from sklearn.feature_extraction.text import CountVectorizer

SPANISH_STOPWORDS = {
    "a", "acá", "ahi", "ahí", "al", "algo", "algunas", "algunos", "ante", "antes", "aquel", "aquella", "aquellas", "aquello", "aquellos",
    "aqui", "aquí", "arriba", "asi", "así", "atras", "aun", "aunque", "bajo", "bien", "cada", "casi", "como", "con", "contra",
    "cual", "cuando", "de", "del", "desde", "donde", "dos", "durante", "e", "el", "él", "ella", "ellas", "ellos", "en", "entre",
    "era", "eran", "eres", "es", "esa", "esas", "ese", "eso", "esos", "esta", "está", "estaba", "estaban", "estado", "estamos", "estan", "están",
    "estar", "este", "esto", "estos", "estoy", "fue", "fueron", "fui", "ha", "haber", "habia", "había", "hace", "hacen", "hacer", "hacia",
    "han", "hasta", "hay", "la", "las", "le", "les", "lo", "los", "mas", "más", "me", "mi", "mis", "mismo", "mucha", "muchas", "mucho", "muchos",
    "muy", "nada", "ni", "no", "nos", "nosotras", "nosotros", "nuestra", "nuestras", "nuestro", "nuestros", "o", "otra", "otras", "otro", "otros",
    "para", "pero", "poco", "por", "porque", "que", "qué", "quien", "quién", "se", "sea", "segun", "según", "ser", "si", "sí", "sin",
    "sobre", "son", "su", "sus", "tambien", "también", "tan", "tanto", "te", "tenia", "tenía", "tener", "tiene", "tienen", "todo", "todos",
    "tu", "tus", "un", "una", "unas", "uno", "unos", "usted", "ustedes", "va", "van", "vez", "y", "ya", "yo",
}

DOMAIN_STOPWORDS = {
    "anfibia", "revista", "cronica", "crónica", "texto", "nota", "parte", "dice", "dijo", "hacer", "hace", "anos", "años",
    "argentina", "argentino", "argentina", "argentinas", "argentinos", "buenos", "aires",
    "ano", "anos", "año", "poder",
    # --- LEMATIZADA PERIODÍSTICA ---
    "poder", "decir", "ir", "ver", "solo", "llegar", "saber", "dar", 
    "querer", "pasar", "volver", "día", "nuevo", "parecer", "quedar", 
    "creer", "dejar", "seguir", "encontrar", "llamar", "venir", "empezar",
    "persona", "historia", "vida", "caso", "momento", "tiempo", "lugar", "forma",
    # Marcadores temporales y conectores narrativos
    "primero", "despues", "después", "dia", "día", "hoy", "ahora", "siempre", "nunca", "mientras", "momento",
    # Sustantivos/Adjetivos genericos
    "cosa", "alguno", "algun", "medio", "propio", "cierto", "grande"
}

STOPWORDS = sorted(SPANISH_STOPWORDS | DOMAIN_STOPWORDS)

documents = df['document_text_final'].astype(str).tolist()
timestamps = df['post_date'].tolist()
timestamps=[timestamp.split(' ')[0] for timestamp in timestamps]
timestamps = pd.to_datetime(timestamps,format="%Y-%m-%d", errors='coerce')#.tolist()

output_dir = PROJECT_ROOT / "outputs" / "bertopic"
output_dir.mkdir(exist_ok=True, parents=True)

# Create your representation model
representation_model = KeyBERTInspired()

topic_model = BERTopic(
    language='spanish',
    calculate_probabilities=True,
    min_topic_size=10,
    nr_topics='auto',
    representation_model=representation_model,
    vectorizer_model=CountVectorizer(stop_words=STOPWORDS),
)
try:
    topic_model.load(output_dir / 'bertopic_model')
    topics, probs = topic_model.transform(documents)
except:
    topics, probs = topic_model.fit_transform(documents)
    topic_model.save(output_dir / 'bertopic_model')

c:\Trabajos\anfibia-unsam\venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
Loading weights: 100%|██████████| 199/199 [00:00<00:00, 3238.10it/s]
2026-08-26 19:59:29,279 - BERTopic - WARNING: When you use `pickle` to save/load a BERTopic model,please make sure that the environments in which you saveand load the model are **exactly** the same. The version of BERTopic,its dependencies, and python need to remain the same.


In [8]:
print(len(documents))

3556


In [9]:
probs.shape

(3556, 28)

In [10]:
topic_info = topic_model.get_topic_info()
assert not topic_info.empty, 'BERTopic no devolvió tópicos'
assert 'Topic' in topic_info.columns, 'La tabla de tópicos no tiene la columna esperada'

topic_info.head()

,Topic,Count,Name,Representation,Representative_Docs
0,-1,1520,-1_político_política_políticas_políticos,"[político, política, políticas, políticos, tal...","[Nuestro futuro. Con el presente en pausa, qué..."
1,0,551,0_político_política_políticas_políticos,"[político, política, políticas, políticos, dem...",[Mercado y pragmatismo. Macri presentó su gabi...
2,1,273,1_literatura_escritores_autor_escritor,"[literatura, escritores, autor, escritor, nove...",[Todos anfibios. A partir de la salida en Chil...
3,2,177,2_feministas_feminismo_feminismos_feminista,"[feministas, feminismo, feminismos, feminista,...",[Conflictos y desafíos en el feminismo. La rev...
4,3,141,3_brasil_fifa_fútbol_futbolistas,"[brasil, fifa, fútbol, futbolistas, equipos, d...",[Los días que fuimos felices. En tres décadas ...


In [11]:
topics_over_time = topic_model.topics_over_time(
    documents,
    timestamps,
    datetime_format="%Y-%m-%d",
    nr_bins=20,
    # evolution_tuning=False,
)

# output_dir = PROJECT_ROOT / "output" / "bertopic_outputs_no_lematizado"
output_dir.mkdir(exist_ok=True, parents=True)
topics_over_time.to_csv(output_dir / 'topics_over_time.csv', index=False)

topics_over_time.head()

,Topic,Words,Frequency,Timestamp
0,-1,"carlos, habían, kaiser, junto, rosita",13,2012-05-08 20:29:45.600
1,0,"político, política, electoral, elecciones, bol...",2,2012-05-08 20:29:45.600
2,1,"literaria, literatura, escritores, libro, autor",2,2012-05-08 20:29:45.600
3,2,"gays, queer, gay, fetichismo, sexualidad",2,2012-05-08 20:29:45.600
4,3,"lorenzo, brasil, adolfo, entrenador, campeón",4,2012-05-08 20:29:45.600


In [12]:
import pandas as pd
import plotly.express as px

analysis_df = df[['document_text_final', 'post_date']].copy()
analysis_df['post_date'] = pd.to_datetime(analysis_df['post_date'], errors='coerce')

topic_timeline = []

category_documents = analysis_df['document_text_final'].astype(str).tolist()
category_topics, _ = topic_model.transform(category_documents)

per_month = (
    pd.DataFrame({
        'topic': category_topics,
        'date': analysis_df['post_date'],
    })
    .assign(date=lambda x: x['date'].dt.to_period('M').dt.to_timestamp())
    .groupby(['date',  'topic'])
    .size()
    .reset_index(name='count')
)
topic_timeline.append(per_month)

topics_all_categories = pd.concat(topic_timeline, ignore_index=True)
topics_all_categories = topics_all_categories[topics_all_categories['topic'] != -1].copy()


In [13]:
top_topics = (
    topics_all_categories.groupby('topic')['count'].sum().sort_values(ascending=False).head(6).index.tolist()
)
plot_df = topics_all_categories[topics_all_categories['topic'].isin(top_topics)].copy()

fig = px.line(
    plot_df,
    x='date',
    y='count',
    color='topic',
    # line_dash='topic',
    markers=True,
    title='Evolución de tópicos por categoría',
)
fig.update_layout(height=500, xaxis_title='Mes', yaxis_title='Cantidad de documentos')
fig

## 3. Evolución de tópicos por categoría

Este bloque muestra la evolución temporal de los tópicos agrupada también por categoría para comparar cómo cambian los temas entre secciones del corpus.

In [14]:
import pandas as pd
import plotly.express as px

analysis_df = df[['document_text_final', 'post_date', 'category']].copy()
analysis_df['category'] = categories_df['category_normalized']
analysis_df['category'] = analysis_df['category'].fillna('sin_categoria').astype(str).str.strip()
analysis_df['category'] = analysis_df['category'].replace('', 'sin_categoria')
analysis_df['post_date'] = pd.to_datetime(analysis_df['post_date'], errors='coerce')

category_topic_timeline = []
for category_name, group in analysis_df.groupby('category', dropna=False):
    if group.empty:
        continue

    category_documents = group['document_text_final'].astype(str).tolist()
    category_topics, _ = topic_model.transform(category_documents)

    per_month = (
        pd.DataFrame({
            'category': category_name,
            'topic': category_topics,
            'date': group['post_date'],
        })
        .assign(date=lambda x: x['date'].dt.to_period('M').dt.to_timestamp())
        .groupby(['date', 'category', 'topic'])
        .size()
        .reset_index(name='count')
    )
    category_topic_timeline.append(per_month)

topics_by_category = pd.concat(category_topic_timeline, ignore_index=True)
topics_by_category = topics_by_category[topics_by_category['topic'] != -1].copy()

# output_dir = PROJECT_ROOT / "output" / "bertopic_outputs_no_lematizado"
output_dir.mkdir(exist_ok=True, parents=True)
topics_by_category.to_csv(output_dir / 'topics_over_time_by_category.csv', index=False)

topics_by_category.head()

,date,category,topic,count
0,2012-12-01,Ambiente,7,1
2,2014-12-01,Ambiente,1,1
3,2015-01-01,Ambiente,1,1
7,2015-10-01,Ambiente,0,1
8,2015-10-01,Ambiente,22,1


In [15]:
top_topics = (
    topics_by_category.groupby('topic')['count'].sum().sort_values(ascending=False).head(6).index.tolist()
)
plot_df = topics_by_category[topics_by_category['topic'].isin(top_topics)].copy()

fig = px.line(
    plot_df,
    x='date',
    y='count',
    color='category',
    line_dash='topic',
    markers=True,
    title='Evolución de tópicos por categoría',
)
fig.update_layout(height=500, xaxis_title='Mes', yaxis_title='Cantidad de documentos')
fig

Para ejemplificar la riqueza de una categoría particular, aislamos una sección específica y calculamos la proporción de documentos que pertenecen a cada tema principal mediante un gráfico de dona.

In [16]:
display(category_counts)

,category,documents
0,Política y Economía,1425
1,Culturas,1038
2,Políticas de géneros,539
3,Otras,165
4,Salud,126
5,Ambiente,113
6,Tecnologías,109
7,Educación,90
8,Amor y sexo,59


In [17]:
import os
import plotly.express as px

OUTPUT_DIR = output_dir / "graficos"
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
...


# 1. Elegir categoría 
CATEGORIA_EJEMPLO = "Políticas de géneros" 

# 2. Filtramos la categoría y sumamos el volumen de cada tópico
df_cat = topics_by_category[topics_by_category['category'] == CATEGORIA_EJEMPLO].copy()

if df_cat.empty:
    print(f"Ojo: La categoría '{CATEGORIA_EJEMPLO}' no existe o no tiene datos.")
else:
    topic_weights = df_cat.groupby('topic')['count'].sum().sort_values(ascending=False)

    # 3. Agarramos los top 7 tópicos 
    top_topics = topic_weights.head(7)

    # 4. Nombres legibles
    labels = []
    for t in top_topics.index:
        name_raw = topic_model.get_topic_info(t)['Name'].values[0]
        clean_name = ", ".join(name_raw.split("_")[1:4]).capitalize()
        labels.append(clean_name)

    # 5. Graficamos
    fig_pie = px.pie(
        names=labels,
        values=top_topics.values,
        title=f"<b>Peso de Tópicos en la categoría: {CATEGORIA_EJEMPLO}</b>",
        hole=0.4, 
        color_discrete_sequence=px.colors.qualitative.Pastel
    )

    # 6. El porcentaje
    fig_pie.update_traces(
        textposition='inside', 
        textinfo='percent',
        insidetextfont=dict(size=14)
    )
    
    fig_pie.update_layout(
        plot_bgcolor='rgba(0,0,0,0)', 
        paper_bgcolor='#ffffff',
        font=dict(size=14),
        showlegend=True
    )

    # 7. Mostrar y exportar
    fig_pie.write_html(f"{OUTPUT_DIR}/03_peso_topicos_{CATEGORIA_EJEMPLO.replace(' ', '_')}.html")
    # fig_pie.write_image(f"{OUTPUT_DIR}/03_peso_topicos_{CATEGORIA_EJEMPLO.replace(' ', '_')}.png")
    fig_pie.show()

In [18]:
for category_name in category_counts["category"]:
    df_cat = topics_by_category[topics_by_category['category'] == category_name].copy()

    if df_cat.empty:
        print(f"Ojo: La categoría '{CATEGORIA_EJEMPLO}' no existe o no tiene datos.")
    else:
        topic_weights = df_cat.groupby('topic')['count'].sum().sort_values(ascending=False)

        # 3. Agarramos los top 7 tópicos 
        top_topics = topic_weights.head(7)    
        # Generate standard BERTopic barchart for these topics at time 'ts'
        fig = topic_model.visualize_barchart(
            topics=top_topics.index,
            n_words=8,
            title=f"<b>Tópicos representativos para {category_name}</b>"
        )
        colors = px.colors.qualitative.G10
        for i, trace in enumerate(fig.data):
            trace.marker.color = colors[i % len(colors)]

        # 3. Limpiar hover, ejes y títulos
        fig.update_traces(hovertemplate="<b>Término:</b> %{y}<br><b>Relevancia:</b> %{x:.3f}<extra></extra>")
        fig.for_each_xaxis(lambda ax: ax.update(title_text="", showticklabels=False))
        fig.for_each_annotation(lambda a: a.update(text=f"<b>{a.text.split('_')[0]}</b>"))
        fig.update_layout(height=750, width=1300, plot_bgcolor='rgba(0,0,0,0)', paper_bgcolor='#ffffff')

        fig.show()

In [19]:
top_topics

topic
0     6
2     4
7     3
10    2
6     1
3     1
14    1
Name: count, dtype: int64

## 4. Validación y gráficos de chequeo

Estas visualizaciones ayudan a revisar si los tópicos son estables y si el modelo captura cambios temporales plausibles.

In [20]:
import os
import plotly.express as px

# 0. Apuntar a la raíz del proyecto
OUTPUT_DIR = output_dir / "graficos"
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

# 1. Generar la figura base
fig = topic_model.visualize_barchart(
    top_n_topics=10,
    n_words=8,
    title="<b>Palabras más representativas por Tópico</b>"
)

# 2. Aplicar paleta de colores por tópico
colors = px.colors.qualitative.G10
for i, trace in enumerate(fig.data):
    trace.marker.color = colors[i % len(colors)]

# 3. Limpiar hover, ejes y títulos
fig.update_traces(hovertemplate="<b>Término:</b> %{y}<br><b>Relevancia:</b> %{x:.3f}<extra></extra>")
fig.for_each_xaxis(lambda ax: ax.update(title_text="", showticklabels=False))
fig.for_each_annotation(lambda a: a.update(text=f"<b>{a.text.split('_')[0]}</b>"))
fig.update_layout(height=750, width=1300, plot_bgcolor='rgba(0,0,0,0)', paper_bgcolor='#ffffff')

# 4. Guardar en el output correcto
output_file = OUTPUT_DIR / "01_barchart_topicos_hover.html"
fig.write_html(output_file)

output_file = f"{OUTPUT_DIR}/01_barchart_topicos_hover.png"
# fig.write_image(output_file)

fig.show()

Este gráfico traza la frecuencia y relevancia relativa de los 10 tópicos principales a lo largo del corpus
- Los marcadores ($\diamond$) señalan el punto máximo de publicación para cada tópico 
- Por defecto se visualizan los 3 tópicos de mayor volumen

In [21]:
import os
import plotly.express as px
import plotly.graph_objects as go

# Crear carpeta de salida si no existe
OUTPUT_DIR = output_dir / "graficos"
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)


# Generar gráfico base de tópicos en el tiempo
fig = topic_model.visualize_topics_over_time(topics_over_time, top_n_topics=10)
colors = px.colors.qualitative.G10

# Iterar sobre cada tópico para formatear líneas y marcar picos
for i, t in enumerate(list(fig.data)):
    c = colors[i % len(colors)]
    vis = True if i < 3 else 'legendonly'  # Mostrar solo el top 3 por defecto
    name = t.name.replace("_", " ")        # Limpiar formato de nombre para la leyenda

    # Estilizar línea del tópico y hover individual
    t.update(name=name, visible=vis, legendgroup=f"g{i}", line=dict(color=c, width=3),
             hovertemplate=f"<b>{name}</b><br>Fecha: %{{x}}<br>Frecuencia: %{{y}}<extra></extra>")
    
    # Calcular e insertar el pico global (máximo)
    idx = list(t.y).index(max(t.y))
    fig.add_trace(go.Scatter(
        x=[t.x[idx]], y=[t.y[idx]], text=[t.y[idx]], mode="markers+text", textposition="top center",
        showlegend=False, visible=vis, legendgroup=f"g{i}",
        marker=dict(size=10, color=c, symbol="diamond", line=dict(width=1.5, color="#111")),
        hovertemplate=f"<b>Pico ({name})</b><br>Fecha: %{{x}}<br>Máximo: %{{y}}<extra></extra>"
    ))

# Ajustes visuales
fig.update_layout(title="<b>Evolución temporal de Tópicos</b>", plot_bgcolor='rgba(0,0,0,0)', paper_bgcolor='#fff', height=600, hovermode="closest")
fig.update_yaxes(showgrid=True, gridcolor="#ebebeb")

# Exportar HTML y renderizar
fig.write_html(OUTPUT_DIR / "02_evolucion_topicos_picos.html")
# fig.write_image(f"{OUTPUT_DIR}/02_evolucion_topicos_picos.png")
fig.show()

In [22]:
# Verificar que al menos algunos tópicos tengan palabras descriptivas
sample_topics = topic_info['Topic'].dropna().astype(int).tolist()
assert sample_topics, 'No hay índices de tópicos disponibles para inspeccionar'

for topic_id in sample_topics[:5]:
    words = topic_model.get_topic(topic_id)[:5]
    print(topic_id, words)

-1 [('político', np.float32(0.5731757)), ('política', np.float32(0.5727508)), ('políticas', np.float32(0.5620635)), ('políticos', np.float32(0.5384357)), ('tal', np.float32(0.48579076))]
0 [('político', np.float32(0.6101318)), ('política', np.float32(0.60905147)), ('políticas', np.float32(0.592056)), ('políticos', np.float32(0.58975905)), ('democracia', np.float32(0.56274915))]
1 [('literatura', np.float32(0.6146696)), ('escritores', np.float32(0.6062219)), ('autor', np.float32(0.59551203)), ('escritor', np.float32(0.5824758)), ('novela', np.float32(0.5419061))]
2 [('feministas', np.float32(0.7794355)), ('feminismo', np.float32(0.7672067)), ('feminismos', np.float32(0.7657449)), ('feminista', np.float32(0.7451251)), ('patriarcado', np.float32(0.49618316))]
3 [('brasil', np.float32(0.6161517)), ('fifa', np.float32(0.59307677)), ('fútbol', np.float32(0.54926026)), ('futbolistas', np.float32(0.5330804)), ('equipos', np.float32(0.46620488))]


In [23]:
# Ver el resumen de tópicos y sus palabras clave
topic_info = topic_model.get_topic_info()
display(topic_info.head(10))
# Inspeccionar las palabras del tópico principal (ejemplo: tópico 0)
topic_model.get_topic(0)

,Topic,Count,Name,Representation,Representative_Docs
0,-1,1520,-1_político_política_políticas_políticos,"[político, política, políticas, políticos, tal...","[Nuestro futuro. Con el presente en pausa, qué..."
1,0,551,0_político_política_políticas_políticos,"[político, política, políticas, políticos, dem...",[Mercado y pragmatismo. Macri presentó su gabi...
2,1,273,1_literatura_escritores_autor_escritor,"[literatura, escritores, autor, escritor, nove...",[Todos anfibios. A partir de la salida en Chil...
3,2,177,2_feministas_feminismo_feminismos_feminista,"[feministas, feminismo, feminismos, feminista,...",[Conflictos y desafíos en el feminismo. La rev...
4,3,141,3_brasil_fifa_fútbol_futbolistas,"[brasil, fifa, fútbol, futbolistas, equipos, d...",[Los días que fuimos felices. En tres décadas ...
5,4,127,4_muertos_víctimas_muerte_desaparición,"[muertos, víctimas, muerte, desaparición, habí...","[Un karateca, una hiena y cuatro mujeres muert..."
6,5,111,5_escolares_educativo_enseñanza_educativa,"[escolares, educativo, enseñanza, educativa, d...",[Los nuevos mitos de los viejos vinagres. “No ...
7,6,80,6_abortos_aborto_abortar_embarazos,"[abortos, aborto, abortar, embarazos, embarazo...",[Aborto es lengua popular. Este año pasamos de...
8,7,61,7_epidemias_epidemia_coronavirus_contagio,"[epidemias, epidemia, coronavirus, contagio, p...",[Rosario: 14 días para aislar el virus. Adoles...
9,8,58,8_político_políticos_política_libertario,"[político, políticos, política, libertario, li...",[El pueblo mileísta. La Libertad Avanza pasó d...


[('político', np.float32(0.6101318)),
 ('política', np.float32(0.60905147)),
 ('políticas', np.float32(0.592056)),
 ('políticos', np.float32(0.58975905)),
 ('democracia', np.float32(0.56274915)),
 ('oposición', np.float32(0.5083958)),
 ('gobierno', np.float32(0.47649533)),
 ('electoral', np.float32(0.47502232)),
 ('crisis', np.float32(0.4574777)),
 ('candidato', np.float32(0.44966957))]

In [24]:

# Run the visualization with the original embeddings
fig = topic_model.visualize_documents(documents)
colors = px.colors.qualitative.G10


# Ajustes visuales
fig.update_layout(title="<b>Espacio de documentos</b>", plot_bgcolor='rgba(0,0,0,0)', paper_bgcolor='#fff', height=600, hovermode="closest")
fig.update_yaxes(showgrid=True, gridcolor="#ebebeb")

# Exportar HTML y renderizar
fig.write_html(OUTPUT_DIR / "02_espacio_documentos.html")
# fig.write_image(f"{OUTPUT_DIR}/02_espacio_documentos.png")
fig.show()



In [25]:
topics_over_time["Topic"]#.keys()

0      -1
1       0
2       1
3       2
4       3
       ..
370    12
371    15
372    18
373    19
374    21
Name: Topic, Length: 375, dtype: int64

In [26]:
# 1. Compute topics over time
# topics_over_time = topic_model.topics_over_time(documents, topics, timestamps, nr_bins=20)

# 2. Extract unique time steps (timestamps)
all_timestamps = topics_over_time["Timestamp"].unique()

# 3. Loop through time steps to generate and show/save barcharts for top topics at each period
for ts in all_timestamps[[0,-1]]:
    # Filter data for specific timestamp
    sub_data = topics_over_time.loc[topics_over_time["Timestamp"] == ts]
    
    # Extract top topics for this specific time slice
    top_topics = sub_data.sort_values(by="Topic", ascending=False)["Topic"].head(5).tolist()
    
    # Generate standard BERTopic barchart for these topics at time 'ts'
    fig = topic_model.visualize_barchart(
        topics=top_topics,
        n_words=8,
        title=f"<b>Palabras más representativas por Tópico para {ts}</b>"
    )
    colors = px.colors.qualitative.G10
    for i, trace in enumerate(fig.data):
        trace.marker.color = colors[i % len(colors)]

    # 3. Limpiar hover, ejes y títulos
    fig.update_traces(hovertemplate="<b>Término:</b> %{y}<br><b>Relevancia:</b> %{x:.3f}<extra></extra>")
    fig.for_each_xaxis(lambda ax: ax.update(title_text="", showticklabels=False))
    fig.for_each_annotation(lambda a: a.update(text=f"<b>{a.text.split('_')[0]}</b>"))
    fig.update_layout(height=750, width=1300, plot_bgcolor='rgba(0,0,0,0)', paper_bgcolor='#ffffff')

    fig.show()


# output_file = f"{OUTPUT_DIR}/01_barchart_topicos_hover.html"

# fig.write_html(output_file)
# output_file = f"{OUTPUT_DIR}/01_barchart_topicos_hover.png"
# # fig.write_image(output_file)

# fig.show()